# 04 - Evaluation

Runs JailbreakBench, HarmBench, AdvBench (ASR), XSTest (over-refusal), MMLU and GSM8K (utility).

**Time estimate:** 1-2 hours on 2x T4 (generation is the bottleneck).

In [ ]:
import os
import sys
REPO_DIR = '/kaggle/working/pc-sae'
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

from src.utils import setup_hf_auth
setup_hf_auth()

In [ ]:
!python scripts/04_evaluate.py

In [ ]:
import json
import pandas as pd
from src.utils import METRICS, load_json

summary = load_json(METRICS / 'evaluation_summary.json')
for k, v in summary.items():
    print(f'  {k}: {v}')

In [ ]:
full = load_json(METRICS / 'evaluation_full.json')
rows = []
for bench in ['jailbreakbench', 'harmbench', 'advbench']:
    if bench in full:
        rows.append({
            'benchmark': bench,
            'n': full[bench].get('n_prompts'),
            'ASR': full[bench].get('attack_success_rate'),
            'monitor_block_rate': full[bench].get('monitor_block_rate'),
            'model_refusal_rate': (full[bench].get('n_refused_by_model', 0)
                                    / max(full[bench].get('n_prompts', 1), 1)),
        })
pd.DataFrame(rows)

In [ ]:
!cat outputs/logs/evaluate.log 2>/dev/null | tail -50
print('\n--- ERRORS ---')
!cat outputs/logs/errors.log 2>/dev/null | tail -30